Here we test some evaluation metrics on the different files we have in our predictions folder, so comparing how the model performance changes depending on the type of test set and the model we used. 

Intial approach: Use the given span_f1 for all files 

Taha's brain: A metric depending on the specific tag we changed so for ex. If we changed the names to names from different regions, then how evaluating on just B-PER and I-PER predictions, whereas if we changed the locations then evaluating only on the B-LOC I-LOC tags, then we have things like random strings and typos there we can just compare with a simple f1 score.

An example metric here could be f1 on that specific tag

The upside of the initial approach is it is universally comparable, easy to code (code is already there just need to make adjustments because our gold and predictions are in the same file (easy peezy))

The second approach might be better but is probably a bit harder to write the code for.

Below I will do the span f1 on the original test set

In [10]:
def readNlu(path):
    """Reads given iob2 file based on path, returns gold truth and predictions as seperate lists
    ----------
    path : str
        path to an iob2 file where the second column is the ground truth and the third column is the predictions
    
    Returns
    ----------
    annotations : list
        a list with all the ground truths where each list within is a seperate sentence
    
    predicition : list
        a list with all the predictions where each list within is a seperate sentence
    """
    annotations = []
    cur_annotation = []

    prediction = []
    cur_prediction = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()
        if line == '':
            annotations.append(cur_annotation)
            cur_annotation = []

            prediction.append(cur_prediction)
            cur_prediction = []
        elif line[0] == '#' and len(line.split(' ')) == 1:
            continue
        else:
            cur_annotation.append(line.split(' ')[1])
            cur_prediction.append(line.split(' ')[2])
    return annotations, prediction

In [36]:
an, pr = readNlu("../predictions/original_test/mono/test_conll_results_mono.iob2")

In [150]:
def read_iob2_file(path):
    """
    Read provided Universal NER iob2 file
    
    :param path: path to read from
    :returns: list with sequences of words and NER labels for each sentence
    """
    data = []
    gold_ner_tags = []
    predicted_ner_tags = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()

        if line:
            if line[0] == '#':
                continue # skip comments
            tok = line.split(' ')
            #print(tok)
            gold_ner_tags.append(tok[1])
            predicted_ner_tags.append(tok[2])
        else:
            if gold_ner_tags:  # skip empty lines
                data.append((gold_ner_tags, predicted_ner_tags))
            gold_ner_tags = []
            predicted_ner_tags = []

    # check for last one
    if gold_ner_tags != []:
        data.append((gold_ner_tags, predicted_ner_tags))
    return data

d = read_iob2_file("../predictions/original_test/mono/test_conll_results_mono.iob2")

In [151]:
len(an) == len(pr) == len(d)

True

FUNCTION CHANGED SUCCESFULLY, Time to test the whole thing

In [16]:
#all the other functions are the same

def toSpans(tags):
    # Converts a list of tags to a list of spans
    # in: ['B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O']
    # out: {'7-9:ORG', '0-2:PER'}
    spans = set()
    for beg in range(len(tags)):
        if tags[beg][0] == 'B':
            end = beg
            for end in range(beg+1, len(tags)):
                if tags[end][0] != 'I':
                    break
            spans.add(str(beg) + '-' + str(end) + ':' + tags[beg][2:])
    return spans

def getBegEnd(span):
    return [int(x) for x in span.split(':')[0].split('-')]

def getLooseOverlap(spans1, spans2):
    # returns the overlap of spans without taking the exact boundaries
    # into account. If entities overlap they also count as found.
    found = 0
    for spanIdx, span in enumerate(spans1):
        spanBeg, spanEnd = getBegEnd(span)
        label = span.split(':')[1]
        match = False
        for span2idx, span2 in enumerate(spans2):
            span2Beg, span2End = getBegEnd(span2)
            label2 = span2.split(':')[1]
            if label == label2:
                if span2Beg >= spanBeg and span2Beg <= spanEnd:
                    match = True
                if span2End <= spanEnd and span2End >= spanBeg:
                    match = True
        if match:
            found += 1
    return found

def getUnlabeled(spans1, spans2):
    # Counts the overlap in spans after removing the labels
    return len(set([x.split(':')[0] for x in spans1]).intersection([x.split(':')[0] for x in spans2]))

In [165]:
s = toSpans(an[10])
print(s)
s = {i for i in s if i.split(":")[1] != "MISC"}
print(s)

{'0-2:PER', '14-16:PER', '19-20:MISC', '23-25:PER'}
{'0-2:PER', '14-16:PER', '23-25:PER'}


In [166]:
#Essentially what's at the end of span_f1.py, a strict scoring system where all tags should match, 
#a loose system where even if one of the bios tag is seen so if IT Univerisity of Copenhagen and only Univerisity is detceted
#then there is still credit given, lastly the unlabelled scoring where if an entity is matched regardless of what it is,
#credit is then given, this will be especially useful in the random strings test set

def evaluate_specific(file_path,ner_tag_measured):
    """Takes a file_path and the ner_tag that was modified in that specific set, then returns the metrics which only look 
    at the models performance on that specific tag
        ----------
    file_path : str
        path to the file you want to measure the performance over 
    
    ner_tag_measured : str
        The group which that specific file was measuring without the B or I so just "PER" or "MISC"

    Returns
    -------
    metrics: int
        a strict precison, recall, f1 score (first 3 numbers), if ner_tag specified then only for those tags
        a loose precison, recall, f1 score (second 3 numbers), if ner_tag specified then only for those tags
        an unlabelled precison, recall, f1 score (last 3 numbers), over the entire dataset regardless of ner_tag
    """
    gold_ners, pred_ners = readNlu(file_path)

    tp = 0
    fp = 0
    fn = 0

    recall_loose_tp = 0
    recall_loose_fn = 0
    precision_loose_tp = 0
    precision_loose_fp = 0

    tp_ul = 0
    fp_ul = 0
    fn_ul = 0 

    for gold_ner, pred_ner in zip(gold_ners, pred_ners):
        gold_spans = toSpans(gold_ner)
        pred_spans = toSpans(pred_ner)

        overlap_ul = getUnlabeled(gold_spans, pred_spans)
        tp_ul += overlap_ul
        fp_ul += len(pred_spans) - overlap_ul
        fn_ul += len(gold_spans) - overlap_ul
        
        if ner_tag_measured:
            gold_spans = {i for i in gold_spans if i.split(":")[1] != ner_tag_measured}
            pred_spans = {i for i in pred_spans if i.split(":")[1] != ner_tag_measured}

        overlap = len(gold_spans.intersection(pred_spans))
        tp += overlap
        fp += len(pred_spans) - overlap
        fn += len(gold_spans) - overlap

        overlap_loose = getLooseOverlap(gold_spans, pred_spans)
        recall_loose_tp += overlap_loose
        recall_loose_fn += len(gold_spans) - overlap_loose

        overlap_loose = getLooseOverlap(pred_spans, gold_spans)
        precision_loose_tp += overlap_loose
        precision_loose_fp += len(pred_spans) - overlap_loose


    prec = 0.0 if tp+fp == 0 else tp/(tp+fp)
    rec = 0.0 if tp+fn == 0 else tp/(tp+fn)
    f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    l_prec = 0.0 if precision_loose_tp + precision_loose_fp == 0 else precision_loose_tp/(precision_loose_tp+precision_loose_fp)
    l_rec = 0.0 if recall_loose_tp+recall_loose_fn == 0 else recall_loose_tp/(recall_loose_tp+recall_loose_fn)
    l_f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    tp = tp_ul
    fp = fp_ul
    fn = fn_ul

    ul_prec = 0.0 if tp+fp == 0 else tp/(tp+fp)
    ul_rec = 0.0 if tp+fn == 0 else tp/(tp+fn)
    ul_f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    return prec, rec, f1, l_rec, l_prec, l_f1, ul_rec, ul_prec, ul_f1

        

In [80]:
import os 

directory = "../predictions"
folders = os.listdir(directory)

meta_data = []
for folder in folders:
    models = os.listdir(os.path.join(directory,folder))
    for model in models:
        files = os.listdir(os.path.join(directory,folder,model))
        for file in files:
            path = os.path.join(directory,folder,model,file)
            sub_category = file.split("_")[0]

            meta_data.append({
                "folder" : folder,
                "model" : model,
                "file" : file,
                "path" : path,
                "sub_categories" : sub_category
            })

df = pd.DataFrame(meta_data)

cat_2_ner = {'gender_names': 'PER',
 'location_exonym_endonym': 'LOC',
 'person': 'PER',
 'pronouns': False,
 'random': False,
 'original_test': False,
 'typos': False,
 'typos_entity' : 'PER',
 'typos_loc' : 'LOC'}

df['modified_ner'] = df["folder"].map(cat_2_ner)

df.to_csv("../file_metadata.csv")



In [81]:
from span_f1_adjusted import evaluate_specific

metadata = pd.read_csv("../file_metadata.csv")

results_data = []
for i in range(len(metadata)):
    file_path, modifed_ner = metadata["path"].iloc[i], metadata["modified_ner"].iloc[i]
    folder, model, file, sub_categories = metadata["folder"].iloc[i], metadata["model"].iloc[i], metadata["file"].iloc[i], metadata["sub_categories"].iloc[i]

    prec, rec, f1, l_rec, l_prec, l_f1, ul_rec, ul_prec, ul_f1 = evaluate_specific(file_path,modifed_ner)

    results_data.append({
        "folder" : folder,
        "model" : model,
        "modified_ner" : modifed_ner,
        "sub_categories" : sub_categories,
        "file" : file,
        "path" : file_path,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loose_precision": l_prec,
        "loose_recall": l_rec,
        "loose_f1": l_f1,
        "unlabeled_precision": ul_prec,
        "unlabeled_recall": ul_rec,
        "unlabeled_f1": ul_f1
        })

results = pd.DataFrame(results_data)

results.to_csv("../evaluation_results.csv")

Someone else can analyze the results Peace out

In [53]:
df = pd.read_csv("../evaluation_results.csv")

In [67]:
original_mask = ((df["sub_categories"] == "test"))
base_results = df[original_mask].groupby("file").mean(numeric_only=True)
base_results

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,,
test_conll_results_mono.iob2,44.0,0.673139,0.662890,0.667975,0.869831,0.860482,0.667975,0.710356,0.699540,0.667975
test_conll_results_multi.iob2,45.0,0.665510,0.577373,0.618316,0.885714,0.764518,0.618316,0.699796,0.607118,0.618316


In [68]:
gender_mono_mask = ((df["folder"] == "gender_names") & (df["model"] == "mono"))
gender_mono_results = df[gender_mono_mask].groupby(df["sub_categories"]).mean(numeric_only=True)
gender_mono_results

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,,
female,4.5,0.683973,0.680799,0.682382,0.836430,0.836715,0.682382,0.724970,0.714341,0.682382
male,14.5,0.686738,0.680452,0.683580,0.839814,0.836343,0.683580,0.724243,0.713792,0.683580


In [69]:
gender_multi_mask = ((df["folder"] == "gender_names") & (df["model"] == "multi"))
gender_multi_results = df[gender_multi_mask].groupby(df["sub_categories"]).mean(numeric_only=True)
gender_multi_results

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,,
female,24.5,0.684694,0.562987,0.617903,0.852201,0.704341,0.617903,0.715034,0.62130,0.617903
male,34.5,0.690256,0.563384,0.620399,0.859006,0.704887,0.620399,0.713586,0.62114,0.620399


In [70]:
pronouns_mono_mask = ((df["folder"] == "pronouns") & (df["model"] == "mono"))
pronouns_mono_results = df[pronouns_mono_mask].groupby("sub_categories").mean(numeric_only=True)
pronouns_mono_results

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,,
female,146.0,0.673260,0.662890,0.668035,0.869808,0.860305,0.668035,0.710484,0.699540,0.668035
male,147.0,0.673198,0.663067,0.668094,0.869854,0.860659,0.668094,0.710228,0.699540,0.668094
neutral,148.0,0.672842,0.662358,0.667559,0.869424,0.859773,0.667559,0.710432,0.699363,0.667559


In [71]:
pronouns_multi_mask = ((df["folder"] == "pronouns") & (df["model"] == "multi"))
pronouns_multi_results = df[pronouns_multi_mask].groupby("sub_categories").mean(numeric_only=True)
pronouns_multi_results

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,,
female,149.0,0.665850,0.577550,0.618565,0.886099,0.764873,0.618565,0.699939,0.607118,0.618565
male,150.0,0.665714,0.577550,0.618506,0.885918,0.764873,0.618506,0.699796,0.607118,0.618506
neutral,151.0,0.665238,0.577018,0.617996,0.885487,0.764164,0.617996,0.699939,0.607118,0.617996


In [73]:
location_mono_mask = ((df["folder"] == "location_exonym_endonym") & (df["model"] == "mono"))
location_mono_result = df[location_mono_mask].groupby("file").mean(numeric_only=True)
location_mono_result

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,,
location_endonym_results_mono.iob2,40.0,0.641687,0.669095,0.655105,0.830843,0.871608,0.655105,0.698439,0.681126,0.655105
location_latin_results_mono.iob2,41.0,0.633530,0.674121,0.653195,0.813459,0.871859,0.653195,0.702942,0.689625,0.653195


In [74]:
location_multi_mask = ((df["folder"] == "location_exonym_endonym") & (df["model"] == "multi"))
location_multi_result = df[location_multi_mask].groupby("file").mean(numeric_only=True)
location_multi_result

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,,
location_endonym_results_multi.iob2,42.0,0.653392,0.556533,0.601085,0.871681,0.736181,0.601085,0.684426,0.591360,0.601085
location_latin_results_multi.iob2,43.0,0.641915,0.562563,0.599625,0.847764,0.736935,0.599625,0.698237,0.603045,0.599625


In [64]:
random_mono_mask = ((df["folder"] == "random") & (df["model"] == "mono"))
random_mono_result = df[random_mono_mask].groupby("sub_categories").mean(numeric_only=True)
random_mono_result

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,,
random,156.5,0.366357,0.186756,0.247391,0.50736,0.256976,0.247391,0.648926,0.330825,0.247391


In [75]:
random_multi_mask = ((df["folder"] == "random") & (df["model"] == "multi"))
random_multi_result = df[random_multi_mask].groupby("sub_categories").mean(numeric_only=True)
random_multi_result

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,,
random,166.5,0.452884,0.16829,0.245382,0.628598,0.234136,0.245382,0.673987,0.25046,0.245382


In [76]:
person_mono_mask = ((df["folder"] == "person") & (df["model"] == "mono"))
person_mono_result = df[person_mono_mask].groupby("sub_categories").mean(numeric_only=True)
person_mono_result

,Unnamed: 0,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,,
african,50.5,0.676313,0.678343,0.677326,0.827485,0.834135,0.677326,0.716844,0.706285,0.677326
american,60.5,0.684415,0.682138,0.683274,0.837468,0.838725,0.683274,0.726499,0.716785,0.683274
arabic,70.5,0.688201,0.683180,0.685681,0.842014,0.840089,0.685681,0.726591,0.716608,0.685681
european,80.5,0.686509,0.678045,0.682251,0.839248,0.833069,0.682251,0.717313,0.705577,0.682251
indian,90.5,0.685702,0.682535,0.684114,0.839451,0.839767,0.684114,0.729288,0.719281,0.684114


In [132]:
person_multi_mask = ((df["folder"] == "person") & (df["model"] == "multi"))
person_multi_result = df[person_multi_mask].groupby("sub_categories").mean(numeric_only=True)
person_multi_result

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,
african,0.668678,0.576735,0.619312,0.874409,0.755205,0.619312,0.711192,0.613403,0.619312
american,0.680142,0.590285,0.632035,0.881750,0.763964,0.632035,0.719178,0.624163,0.632035
arabic,0.684676,0.592351,0.635176,0.885048,0.767440,0.635176,0.721942,0.624593,0.635176
european,0.674745,0.586579,0.627581,0.887393,0.767812,0.627581,0.706212,0.613934,0.627581
indian,0.687131,0.593892,0.637118,0.884116,0.765687,0.637118,0.724844,0.626487,0.637118


In [140]:
df.to_csv("../ner_evaluation_results.csv")

In [78]:
### ADDING TYPOS_LOC to eval_results ###
df = pd.read_csv("../evaluation_results.csv")
df.head()

,id,folder,model,modified_ner,sub_categories,file,path,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
0,0,gender_names,mono,PER,female,female_names_test_0_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.684552,0.680476,0.682508,0.837285,0.836765,0.682508,0.725589,0.714412,0.682508
1,1,gender_names,mono,PER,female,female_names_test_1_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.686078,0.680972,0.683516,0.839290,0.837013,0.683516,0.724969,0.714058,0.683516
2,2,gender_names,mono,PER,female,female_names_test_2_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.683163,0.681469,0.682315,0.835364,0.837261,0.682315,0.725624,0.715475,0.682315
3,3,gender_names,mono,PER,female,female_names_test_3_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.685393,0.680972,0.683176,0.837453,0.836269,0.683176,0.724528,0.713881,0.683176
4,4,gender_names,mono,PER,female,female_names_test_4_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.683803,0.679732,0.681762,0.836536,0.835773,0.681762,0.723810,0.713173,0.681762


In [6]:
import pandas as pd

In [34]:
def evaluate_individuals(file_path,ner_tag_measured):
    """Takes a file_path and the ner_tag that was modified in that specific set, then returns the metrics which only look 
    at the models performance on that specific tag
        ----------
    file_path : str
        path to the file you want to measure the performance over 
    
    ner_tag_measured : str
        The group which that specific file was measuring without the B or I so just "PER" or "MISC"

    Returns
    -------
    metrics: int
        a strict precison, recall, f1 score (first 3 numbers), if ner_tag specified then only for those tags
        a loose precison, recall, f1 score (second 3 numbers), if ner_tag specified then only for those tags
        an unlabelled precison, recall, f1 score (last 3 numbers), over the entire dataset regardless of ner_tag
    """
    gold_ners, pred_ners = readNlu(file_path)

    result = pd.DataFrame([], columns=["tp","fp","fn","recall_loose_tp","recall_loose_fn","precision_loose_tp","precision_loose_fp","tp_ul","fp_ul","fn_ul"])

    for gold_ner, pred_ner in zip(gold_ners, pred_ners):
        new_row = [None]*len(result.columns)

        gold_spans = toSpans(gold_ner)
        pred_spans = toSpans(pred_ner)

        overlap_ul = getUnlabeled(gold_spans, pred_spans)
        new_row[7] = overlap_ul
        new_row[8] = len(pred_spans) - overlap_ul
        new_row[9] = len(gold_spans) - overlap_ul
        
        if ner_tag_measured:
            gold_spans = {i for i in gold_spans if i.split(":")[1] != ner_tag_measured}
            pred_spans = {i for i in pred_spans if i.split(":")[1] != ner_tag_measured}

        overlap = len(gold_spans.intersection(pred_spans))
        new_row[0] = overlap
        new_row[1] = len(pred_spans) - overlap
        new_row[2] = len(gold_spans) - overlap

        overlap_loose = getLooseOverlap(gold_spans, pred_spans)
        new_row[3] = overlap_loose
        new_row[4] = len(gold_spans) - overlap_loose

        overlap_loose = getLooseOverlap(pred_spans, gold_spans)
        new_row[5] = overlap_loose
        new_row[6] = len(pred_spans) - overlap_loose

        result.loc[len(result)] = new_row


    # prec = 0.0 if tp+fp == 0 else tp/(tp+fp)
    # rec = 0.0 if tp+fn == 0 else tp/(tp+fn)
    # f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    # l_prec = 0.0 if precision_loose_tp + precision_loose_fp == 0 else precision_loose_tp/(precision_loose_tp+precision_loose_fp)
    # l_rec = 0.0 if recall_loose_tp+recall_loose_fn == 0 else recall_loose_tp/(recall_loose_tp+recall_loose_fn)
    # l_f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    # tp = tp_ul
    # fp = fp_ul
    # fn = fn_ul

    # ul_prec = 0.0 if tp+fp == 0 else tp/(tp+fp)
    # ul_rec = 0.0 if tp+fn == 0 else tp/(tp+fn)
    # ul_f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    return result

    # return prec, rec, f1, l_rec, l_prec, l_f1, ul_rec, ul_prec, ul_f1

metadata = pd.read_csv("../file_metadata.csv")

file_path = metadata["path"].iloc[0].replace("\\","/")

for i in range(len(metadata)):
    file_path, modifed_ner = metadata["path"].iloc[i], metadata["modified_ner"].iloc[i]

    # support for mac directory paths
    file_path = file_path.replace("\\","/")

    indiv_result = evaluate_individuals(file_path,modifed_ner)


    indiv_result.to_csv(f"../indiv_evaluations/{file_path.split("/")[-3]}/{file_path.split("/")[-2]}/evaluated_{file_path.split("/")[-1].split(".")[0]}.csv", index=False)

    if i % 10 == 0:
        print(round(i/len(metadata),4)*100,"%")

../predictions/gender_names/mono/female_names_test_0_results_mono.iob2
PER
0.0 %
../predictions/gender_names/mono/female_names_test_1_results_mono.iob2
PER
../predictions/gender_names/mono/female_names_test_2_results_mono.iob2
PER
../predictions/gender_names/mono/female_names_test_3_results_mono.iob2
PER
../predictions/gender_names/mono/female_names_test_4_results_mono.iob2
PER
../predictions/gender_names/mono/female_names_test_5_results_mono.iob2
PER
../predictions/gender_names/mono/female_names_test_6_results_mono.iob2
PER
../predictions/gender_names/mono/female_names_test_7_results_mono.iob2
PER
../predictions/gender_names/mono/female_names_test_8_results_mono.iob2
PER
../predictions/gender_names/mono/female_names_test_9_results_mono.iob2
PER
../predictions/gender_names/mono/male_names_test_0_results_mono.iob2
PER
4.31 %
../predictions/gender_names/mono/male_names_test_1_results_mono.iob2
PER
../predictions/gender_names/mono/male_names_test_2_results_mono.iob2
PER
../predictions/gen

KeyboardInterrupt: 